In [ ]:
from youtube_insight_engine.utils.common import read_yaml,createDIr
import requests
import os
from pathlib import Path
import urllib.request as request
import pandas as pd
import zipfile
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import TextVectorization, Embedding, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.models import Sequential


In [5]:
%pwd

'c:\\Users\\NishantHP\\Desktop\\YouTube Audience Insights Engine\\research'

In [6]:
os.chdir('..')

In [7]:
%pwd

'c:\\Users\\NishantHP\\Desktop\\YouTube Audience Insights Engine'

In [35]:
config=read_yaml(Path('config/config.yaml'))
createDIr([config.data_ingestion.rootdir_zip_down])
if not os.path.exists(config.data_ingestion.zip_downl_loc):
    
    filename,header=request.urlretrieve(
        config.data_ingestion.source_url,
        filename=config.data_ingestion.zip_downl_loc)

[2026-04-29 11:51:53,856: INFO: common: yaml file<_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-29 11:51:53,858: INFO: common: created directory at artifacts/data_ingestion]


In [19]:
os.makedirs(config.data_ingestion.unzip_dir_loc, exist_ok=True)
with zipfile.ZipFile(config.data_ingestion.zip_downl_loc, 'r') as zip_ref:
    zip_ref.extractall(config.data_ingestion.unzip_dir_loc)

In [54]:
df=pd.read_csv(config.data_ingestion.unzip_loc)

In [55]:
df

,Comment,Sentiment
0,lets not forget that apple pay in 2014 require...,neutral
1,here in nz 50 of retailers don’t even have con...,negative
2,i will forever acknowledge this channel with t...,positive
3,whenever i go to a place that doesn’t take app...,negative
4,apple pay is so convenient secure and easy to ...,positive
...,...,...
18403,i really like the point about engineering tool...,positive
18404,i’ve just started exploring this field and thi...,positive
18405,excelente video con una pregunta filosófica pr...,neutral
18406,hey daniel just discovered your channel a coup...,positive


In [56]:
df.dropna()

,Comment,Sentiment
0,lets not forget that apple pay in 2014 require...,neutral
1,here in nz 50 of retailers don’t even have con...,negative
2,i will forever acknowledge this channel with t...,positive
3,whenever i go to a place that doesn’t take app...,negative
4,apple pay is so convenient secure and easy to ...,positive
...,...,...
18403,i really like the point about engineering tool...,positive
18404,i’ve just started exploring this field and thi...,positive
18405,excelente video con una pregunta filosófica pr...,neutral
18406,hey daniel just discovered your channel a coup...,positive


In [59]:
comment=df['Comment'].values
sentiments=df['Sentiment'].values

In [61]:
label_encoder=LabelEncoder()
encoded_sentiment=label_encoder.fit_transform(sentiments)

In [62]:
numb_class=len(label_encoder.classes_)

In [64]:
X_train, X_test, y_train, y_test = train_test_split(
    comments, 
    encoded_sentiments, 
    test_size=0.15,      
    random_state=42     
)

In [65]:
vocab_size=20000
max_words=500

In [67]:
vectorize_layer=TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=max_words
)
vectorize_layer.adapt(X_train)


In [75]:
model=Sequential([
    vectorize_layer,
    Embedding(input_dim=vocab_size,output_dim=128,mask_zero=True),
    GlobalAveragePooling1D(),
    Dense(32,activation='relu'),
    Dropout(0.6),
    Dense(numb_class,activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history=model.fit(
    X_train,y_train,
    validation_data=(X_test,y_test),
    epochs=10,
    batch_size=32    
)
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")

Epoch 1/10
488/488 ━━━━━━━━━━━━━━━━━━━━ 15s 29ms/step - accuracy: 0.6788 - loss: 0.7477 - val_accuracy: 0.7459 - val_loss: 0.5947
Epoch 2/10
488/488 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.7799 - loss: 0.5178 - val_accuracy: 0.7648 - val_loss: 0.5726
Epoch 3/10
488/488 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.8566 - loss: 0.3778 - val_accuracy: 0.7568 - val_loss: 0.6346
Epoch 4/10
488/488 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.9065 - loss: 0.2827 - val_accuracy: 0.7477 - val_loss: 0.7086
Epoch 5/10
488/488 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.9334 - loss: 0.2203 - val_accuracy: 0.7481 - val_loss: 0.7999
Epoch 6/10
488/488 ━━━━━━━━━━━━━━━━━━━━ 15s 30ms/step - accuracy: 0.9545 - loss: 0.1684 - val_accuracy: 0.7387 - val_loss: 0.9423
Epoch 7/10
488/488 ━━━━━━━━━━━━━━━━━━━━ 16s 33ms/step - accuracy: 0.9623 - loss: 0.1374 - val_accuracy: 0.7441 - val_loss: 1.0674
Epoch 8/10
488/488 ━━━━━━━━━━━━━━━━━━━━ 17s 34ms/step - accuracy: 0.9683 - loss: 0.1154 - 